In [4]:
#importando bibliotecas 
import re
import time
from collections import Counter
import spacy
from spacy import displacy
from spacy.matcher import Matcher
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy.language import Language

In [5]:
nlp = spacy.load("en_core_web_sm") #carrega o modelo pequeno de inglês

In [6]:
#texto curto
text = """Music is ubiquitous across human cultures—as a source of affective and pleasurable experience, moving us both physically and emotionally—and learning to play music shapes both brain structure and function. Music processing in the brain—namely, perception of melody, harmony, and rhythm—has traditionally been studied as an auditory phenomenon using passive listening paradigms. However, when listening to music, we actively generate predictions about what is likely to happen next. This enactive aspect has led to a more complete understanding of music processing involving brain structures implicated in action, emotion, and learning. Here, we review the cognitive neuroscience literature of music perception. We show that music perception, action, emotion, and learning all rest on the human brain’s fundamental capacity for prediction—as formulated by the predictive coding of music model. This review elucidates how this formulation of music perception and expertise in individuals can be extended to account for the dynamics and underlying brain mechanisms of collective music making. This in turn has important implications for human creativity as evinced by music improvisation. These recent advances shed new light on what makes music meaningful from a neuroscientific perspective."""

In [7]:
#artigo completo
with open(r"C:\Users\marya\Downloads\music_in_the_brain.txt", "r", encoding="utf-8") as f:
    pdf_text = f.read()

In [8]:
pdf_text = pdf_text.replace("-\n", "").replace("\n", " ") #tira as quebras de linha
pdf_text = " ".join(pdf_text.split()) #tira os espaços repetidos
pdf_text = pdf_text.split("Boxes and figure captions")[0] #fica só com o corpo do artigo 
print(len(pdf_text))

62092


In [9]:
doc = nlp(text) #cria o doc

In [10]:
print(len(text)) #quantidade de caracteres
print(len(doc)) #quantidade de tokens

1289
216


In [11]:
print(text.split()[:10]) #split separa só pelo espaço
print([token.text for token in doc[:10]]) #o spacy separa também a pontuação

['Music', 'is', 'ubiquitous', 'across', 'human', 'cultures—as', 'a', 'source', 'of', 'affective']
['Music', 'is', 'ubiquitous', 'across', 'human', 'cultures', '—', 'as', 'a', 'source']


In [12]:
sentences = list(doc.sents) #converte para lista
print(len(sentences))
print(sentences[2])

9
However, when listening to music, we actively generate predictions about what is likely to happen next.


In [13]:
token = sentences[2][9] 
#pega um token da terceira sentença
print(token.text)
print(token.lemma_) #forma base da palavra
print(token.pos_) #classe gramatical
print(token.dep_) #função na frase
print(token.head) #palavra da qual ele depende
print(token.ent_type_) #tipo de entidade

generate
generate
VERB
ROOT
generate



In [14]:
for token in sentences[2]:
    print(token.text, token.pos_, token.dep_)
# classe gramatical e dependência de cada token da sentença

However ADV advmod
, PUNCT punct
when SCONJ advmod
listening VERB advcl
to ADP prep
music NOUN pobj
, PUNCT punct
we PRON nsubj
actively ADV advmod
generate VERB ROOT
predictions NOUN dobj
about ADP prep
what PRON nsubj
is AUX pcomp
likely ADJ acomp
to PART aux
happen VERB xcomp
next ADV advmod
. PUNCT punct


In [15]:
displacy.render(sentences[2], style="dep", jupyter=True) #desenho das dependências

In [16]:
for ent in doc.ents:
    print(ent.text, ent.label_) #entidades do abstract 

In [17]:
doc_pdf = nlp(pdf_text) #doc do artigo completo

In [18]:
for ent in doc_pdf.ents[:15]:
    print(ent.text, ent.label_) #primeiras entidades encontradas no artigo

Review Music ORG
Peter Vuust PERSON
Ole A. Heggli PERSON
Karl J. Friston & PERSON
Pat Metheny PERSON
three CARDINAL
1 CARDINAL
Gregorian GPE
Phil Collins PERSON
Tonight TIME
3 minutes TIME
recent years DATE
1 CARDINAL
PCM ORG
2 CARDINAL


In [20]:
labels = Counter(ent.label_ for ent in doc_pdf.ents) #conta quantas entidades de cada tipo
print(labels.most_common())

[('CARDINAL', 98), ('ORG', 91), ('PERSON', 31), ('DATE', 18), ('NORP', 13), ('GPE', 9), ('PRODUCT', 6), ('ORDINAL', 5), ('LAW', 3), ('TIME', 2), ('QUANTITY', 1), ('LANGUAGE', 1), ('PERCENT', 1), ('LOC', 1)]


In [21]:
orgs = Counter(ent.text for ent in doc_pdf.ents if ent.label_ == "ORG")
print(orgs.most_common(8))
#o modelo marcou siglas como PCM e MMN como organização, isso é erro dele

[('PCM', 22), ('MMN', 11), ('NAcc', 9), ('EEG', 5), ('IFG', 4), ('MEG', 3), ('ears4', 2), ('Cheung', 2)]


In [22]:
exemplo = nlp("Pat Metheny said that listening is the key. Phil Collins played a drum fill in London.")
displacy.render(exemplo, style="ent", jupyter=True)
#entidades coloridas exemplo curto

In [23]:
nlp_md = spacy.load("en_core_web_md")
# modelo com vetores de palavras

In [24]:
doc1 = nlp_md("I enjoy listening to jazz music.")
doc2 = nlp_md("She loves classical songs and concerts.")
doc3 = nlp_md("The car needs new tires and an oil change.")
print(doc1.similarity(doc2)) #nota mais alta
print(doc1.similarity(doc3)) #nota mais baixa

0.8084021806716919
0.5661834478378296


In [25]:
print(nlp.pipe_names) #pipes do modelo na ordem

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [26]:
nlp_blank = spacy.blank("en")
nlp_blank.add_pipe("sentencizer")
#modelo em branco só com o sentencizer 

In [27]:
start = time.time()
d1 = nlp_blank(pdf_text)
print("blank:", round(time.time() - start, 2), "s")
start = time.time()
d2 = nlp(pdf_text)
print("sm:", round(time.time() - start, 2), "s")
# o modelo em branco é bem mais rápido

blank: 0.23 s
sm: 3.06 s


In [28]:
nlp_ruler = spacy.load("en_core_web_sm")
ruler = nlp_ruler.add_pipe("entity_ruler", before="ner")

In [29]:
patterns = [
    {"label": "MODEL", "pattern": "predictive coding of music"},
    {"label": "MODEL", "pattern": "PCM"},
    {"label": "METHOD", "pattern": "EEG"},
    {"label": "METHOD", "pattern": "fMRI"},
    {"label": "ERP", "pattern": "MMN"},
]
ruler.add_patterns(patterns)
#rótulo + o que procurar

In [30]:
doc_ruler = nlp_ruler(text)
for ent in doc_ruler.ents:
    print(ent.text, ent.label_)
#agora ele acha o modelo PCM no abstract

predictive coding of music MODEL


In [31]:
doc_ruler_pdf = nlp_ruler(pdf_text)
siglas = Counter(ent.text for ent in doc_ruler_pdf.ents if ent.label_ in ["MODEL", "METHOD", "ERP"])
print(siglas.most_common())
#conta os termos que eu defini no artigo completo

[('PCM', 24), ('MMN', 12), ('EEG', 6), ('fMRI', 6), ('predictive coding of music', 4)]


In [32]:
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "ADJ"}, {"POS": "NOUN"}] #adjetivo seguido de substantivo
matcher.add("ADJ_NOUN", [pattern])

In [33]:
matches = matcher(doc)
for match_id, start, end in matches:
    print(doc[start:end].text)

human cultures
pleasurable experience
auditory phenomenon
passive listening
enactive aspect
complete understanding
cognitive neuroscience
human brain
fundamental capacity
predictive coding
collective music
important implications
human creativity
recent advances
new light
neuroscientific perspective


In [34]:
matches_pdf = matcher(doc_pdf)
pares = Counter(doc_pdf[start:end].text.lower() for match_id, start, end in matches_pdf)
print(pares.most_common(10)) #pares mais comuns no artigo completo

[('predictive coding', 17), ('musical training', 10), ('predictive processing', 8), ('musical improvisation', 8), ('predictive model', 7), ('predictive brain', 6), ('active inference', 6), ('predictive mechanisms', 5), ('western music', 5), ('musical pleasure', 5)]


In [35]:
matcher2 = Matcher(nlp.vocab)
pattern2 = [{"POS": "ADJ", "OP": "+"}, {"POS": "NOUN", "OP": "+"}]
matcher2.add("ADJ_NOUN", [pattern2], greedy="LONGEST")
for match_id, start, end in matcher2(doc):
    print(doc[start:end].text)

passive listening paradigms
cognitive neuroscience literature
human cultures
pleasurable experience
auditory phenomenon
enactive aspect
complete understanding
human brain
fundamental capacity
predictive coding
collective music
important implications
human creativity
recent advances
new light
neuroscientific perspective


In [36]:
sample = "The MMN peaks 110-250 ms after change onset, and the ERAN peaks at 150-200 ms."
print(re.findall(r"\d{2,3}-\d{2,3} ms", sample))
#findall devolve só os textos

['110-250 ms', '150-200 ms']


In [37]:
for hit in re.finditer(r"\d{2,3}-\d{2,3} ms", sample):
    print(hit.start(), hit.end(), hit.group())
#finditer devolve também a posição de cada um

14 24 110-250 ms
67 77 150-200 ms


In [38]:
doc_regex = doc_pdf.copy()
original_ents = list(doc_regex.ents)
new_ents = []
#copia do doc e lista das entidades que o modelo já achou

In [39]:
for hit in re.finditer(r"\b\d{2,3}(?:-\d{2,3})? ms\b", doc_regex.text):
    span = doc_regex.char_span(hit.start(), hit.end()) #transforma a posição em caracteres em um span de tokens
    if span is not None:
        new_ents.append(Span(doc_regex, span.start, span.end, label="LATENCY"))

In [40]:
doc_regex.ents = filter_spans(new_ents + original_ents) #evita entidades sobrepostas e fica com a mais longa
print([ent.text for ent in doc_regex.ents if ent.label_ == "LATENCY"])

['150-200 ms']


In [41]:
@Language.component("conta_entidades")
def conta_entidades(doc):
    print(len(doc.ents), "entidades")
    return doc #recebe o doc e devolve o doc

In [42]:
nlp_custom = spacy.load("en_core_web_sm")
nlp_custom.add_pipe("conta_entidades", after="ner")
print(nlp_custom.pipe_names)
_ = nlp_custom("Phil Collins played in London in 1981.")

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'conta_entidades']
3 entidades


In [43]:
print([ord(c) for c in "LISTEN"])
print([ord(c) for c in "SILENT"])
#letras viram os mesmos números só em outra ordem por isso usamos palavras

[76, 73, 83, 84, 69, 78]
[83, 73, 76, 69, 78, 84]


In [44]:
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
except Exception:
    from keras_preprocessing.text import Tokenizer
    import numpy as np

    def pad_sequences(sequences, maxlen):
        padded = np.zeros((len(sequences), maxlen), dtype=int)
        for i, seq in enumerate(sequences):
            seq = seq[-maxlen:]
            padded[i, maxlen - len(seq):] = seq
        return padded
#se o tensorflow não tiver o Tokenizer, usa o keras_preprocessing

In [45]:
sentences_video = [
    'i love my dog',
    'I, love my cat',
    'You love my dog!'
]
tokenizer = Tokenizer(num_words=100)
tokenizer.fit_on_texts(sentences_video)
word_index = tokenizer.word_index
print(word_index)
#tudo vira minúsculo a pontuação some e os números seguem a frequência

{'love': 1, 'my': 2, 'i': 3, 'dog': 4, 'cat': 5, 'you': 6}


In [46]:
sentences = [sent.text for sent in doc.sents] #agora com as sentenças do abstract
tokenizer = Tokenizer(num_words=200, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
print(len(word_index))
print(list(word_index.items())[:12])

120
[('<OOV>', 1), ('music', 2), ('and', 3), ('of', 4), ('to', 5), ('the', 6), ('in', 7), ('perception', 8), ('this', 9), ('human', 10), ('a', 11), ('learning', 12)]


In [47]:
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences[0]) #a primeira sentença transformada em números

[2, 16, 28, 29, 10, 30, 11, 31, 4, 32, 3, 33, 34, 35, 36, 17, 37, 3, 38, 12, 5, 39, 2, 40, 17, 13, 41, 3, 42]


In [44]:
test_data = ["Music shapes the brain", "Jazz improvisation is creative"]
print(tokenizer.texts_to_sequences(test_data)) #palavras que não estavam no abstract viram 1 (<OOV>)

[[2, 40, 6, 13], [1, 109, 16, 1]]


In [45]:
padded = pad_sequences(sequences, maxlen=40)
print(padded.shape)
print(padded[0])
# deixa todas as sentenças com o mesmo tamanho (zeros à esquerda)

(9, 40)
[ 0  0  0  0  0  0  0  0  0  0  0  2 16 28 29 10 30 11 31  4 32  3 33 34
 35 36 17 37  3 38 12  5 39  2 40 17 13 41  3 42]
